# try-except-solve — ex2: list-batched safe solve returning list[Optional[Tensor]]

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `try-except-solve`. Running the final beacon cell reports progress against the `LinAlg: try/except solve` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `LinAlg: try/except solve` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`try-except-solve`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "try-except-solve"
DD_SUBTOPIC = "LinAlg: try/except solve"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## try/except solve — list-batched form

Ex1 wrapped ONE call in `try/except RuntimeError` and returned `None` on failure. The natural extension is a PYTHON-LEVEL batch: you have a `list[Tensor]` of `(n, n)` matrices and a parallel `list[Tensor]` of RHS vectors, and you want one solution-or-None per pair without letting one bad matrix kill the rest:

```python
def safe_solve_list(As, bs):
    out = []
    for A, b in zip(As, bs):
        try:
            out.append(t.linalg.solve(A, b))
        except RuntimeError:
            out.append(None)
    return out
```

**Why a Python loop, not `t.linalg.solve` on a stacked tensor.** A stacked solve fails ALL slices the moment one is singular. The loop form is the right tool when failures are per-item and you want surviving results to come through.

**Contrast with `singular-matrix-mask-trick`.** The mask trick stays fully vectorized but always returns a same-shape tensor (NaN-marked at bad slices). The list-of-Optional form is for the smaller-N case where Python overhead is fine and the caller wants explicit per-slot None.

### Exercise 2 — list-batched safe solve returning list[Optional[Tensor]]

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply a `try/except RuntimeError` inside a Python loop over paired `(A, b)` lists to produce a `list[Optional[Tensor]]` where each None marks a singular input without aborting the rest of the batch.
> Keywords: try-except, linalg-solve, list-batch, graceful-failure
> ```

**KCs targeted:** `per-item-try-except-loop`, `return-none-on-singular`

Implement `ex2_safe_solve_list(As, bs)`.

- `As`: a `list[Tensor]` of `(n, n)` matrices (n may differ   across list items).
- `bs`: a parallel `list[Tensor]` of RHS vectors.
- Return `list[Optional[Tensor]]` of the same length: each   entry is either `t.linalg.solve(A, b)` or `None` if that   particular solve raised `RuntimeError`.

Constraints:
- One singular matrix must NOT abort the rest of the loop.
- Output length must equal input length.
- Order preserved (entry `i` of output corresponds to   `(As[i], bs[i])`).
- Don't try to fix singular inputs — just record None.

In [ ]:
from typing import Optional

def ex2_safe_solve_list(As: list, bs: list) -> list:
    """Per-item safe solve. Returns list[Optional[Tensor]]."""
    raise NotImplementedError()


def _test_ex2():
    # Three solves: well-conditioned, singular, well-conditioned.
    A_good = t.tensor([[3.0, 1.0], [1.0, 2.0]])
    A_sing = t.tensor([[1.0, 2.0], [2.0, 4.0]])   # rank-1
    A_id   = t.eye(3)
    b_good = t.tensor([9.0, 8.0])
    b_sing = t.tensor([3.0, 6.0])
    b_id   = t.tensor([1.0, -2.0, 5.0])
    out = ex2_safe_solve_list([A_good, A_sing, A_id], [b_good, b_sing, b_id])
    assert isinstance(out, list), f'must return a list, got {type(out).__name__}'
    assert len(out) == 3, f'len must be 3, got {len(out)}'

    # Entry 0: well-conditioned -> tensor solution.
    assert out[0] is not None, 'well-conditioned must not be None'
    assert t.allclose(A_good @ out[0], b_good, atol=1e-5)
    # Entry 1: singular -> None.
    assert out[1] is None, f'singular must be None, got {out[1]!r}'
    # Entry 2: identity -> b unchanged.
    assert out[2] is not None
    assert t.allclose(out[2], b_id, atol=1e-6)

    # Order preservation: shuffle inputs, output must follow.
    out2 = ex2_safe_solve_list([A_sing, A_good, A_sing], [b_sing, b_good, b_sing])
    assert out2[0] is None
    assert out2[1] is not None and t.allclose(A_good @ out2[1], b_good, atol=1e-5)
    assert out2[2] is None

    # Empty input -> empty list (loop boundary).
    assert ex2_safe_solve_list([], []) == []

    # All-singular input -> all-None output.
    out_all_bad = ex2_safe_solve_list([A_sing, A_sing], [b_sing, b_sing])
    assert out_all_bad == [None, None]

    # Heterogeneous sizes: a 2x2 followed by a 4x4 should both succeed.
    A4 = t.eye(4) * 2 + t.ones(4, 4) * 0.05
    b4 = t.tensor([1.0, 2.0, 3.0, 4.0])
    out_mixed = ex2_safe_solve_list([A_good, A4], [b_good, b4])
    assert out_mixed[0] is not None and out_mixed[0].shape == (2,)
    assert out_mixed[1] is not None and out_mixed[1].shape == (4,)
    assert t.allclose(A4 @ out_mixed[1], b4, atol=1e-4)

    # Singular in the MIDDLE must not break the loop on the tail.
    out_mid = ex2_safe_solve_list(
        [A_good, A_sing, A_good, A_sing, A_id],
        [b_good, b_sing, b_good, b_sing, b_id],
    )
    assert [x is None for x in out_mid] == [False, True, False, True, False]
    _dd_passed.add('ex2')
    print("ex2 ✓")

_test_ex2()

<details><summary>Solution</summary>

```python
from typing import Optional

def ex2_safe_solve_list(As, bs):
    out = []
    for A, b in zip(As, bs):
        try:
            out.append(t.linalg.solve(A, b))
        except RuntimeError:
            out.append(None)
    return out
```

**Why the try lives INSIDE the loop.** A single `try` around the whole loop would abort on the first singular matrix — exactly the behavior we're trying to avoid. Per-iteration `try/except` is the unit of graceful-failure here.

**Why `zip(As, bs)`.** Mixed sizes per iteration means you can't stack into a batched tensor anyway — the list loop is the natural shape. If sizes WERE uniform you'd want the `singular-matrix-mask-trick` atom instead.

**Python overhead is fine here.** N is small (a few hundred at most). The savings from vectorizing don't offset the API ugliness of NaN-marked tensors. Use the list-of-Optional form when N < ~10k.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()